# Preset 參考圖（Colab / SDXL base 1.0，免登入）

用 Colab GPU 替 Prompt Builder 的 313 個 preset 逐一生圖，作為第二家參考圖（來源標示 `colab`）。

**為什麼是 SDXL**：FLUX.1-schnell 在 Hugging Face 是受限模型，要登入並分享聯絡資訊（2026-10-03 實測 401）。本筆記本改用 `stabilityai/stable-diffusion-xl-base-1.0`：**免登入、不需 token、不分享任何資料**（2026-10-03 以 HF API 查證 gated=False），不花錢，只用 Colab 運算單位。

**用法**：執行階段選 **T4 GPU** → 依序執行 → 先看「煙霧測試」再跑「批次」。
**可續跑**：進度記在 Drive 的 `manifest.json`，斷線後重跑「批次」即可。
**輸入**：`presets_prompts.json`；選用 `agy_manifest.json`（跳過 AGY 已有圖者）。
**輸出**（Drive `prompt_builder_colab/`）：`preset-originals/colab/*.png`（原始尺寸、不縮放、無浮水印）與 `preset-previews/colab/manifest.json`（結構同 Codex／AGY）。

**限制**：
- 開源模型，風格與品質與 Gemini／Codex 不同；中文招牌與海報文字多半亂碼。
- SDXL 的 CLIP 文字上限 77 tokens：送進模型的是**清掉 markdown 記號後的 prompt**（原 prompt 的 `prompt_sha256` 不變），超過 77 tokens 的部分仍會被截斷，manifest 標 `truncated: true`。
- 免費 T4 時數不固定；速度與顯存未在你帳號實測，以煙霧測試為準。

In [ ]:
# 1) 環境檢查與安裝
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout or "找不到 GPU：請把執行階段類型改成 T4 GPU")
subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","diffusers","transformers","accelerate","safetensors"])

In [ ]:
# 2) 設定、掛載 Drive、取得輸入檔
import os, json, hashlib, re, time
from google.colab import drive, files
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/prompt_builder_colab"
ORIG = f"{ROOT}/preset-originals/colab"
PREV = f"{ROOT}/preset-previews/colab"
MANIFEST = f"{PREV}/manifest.json"
os.makedirs(ORIG, exist_ok=True); os.makedirs(PREV, exist_ok=True)

MAX_PIXELS = 1024 * 1024   # 目標像素數；顯存不足可降到 768*768
MAX_NEW_PER_RUN = 40       # 每次執行最多新生成幾張（控制運算單位消耗）
MAX_CONSEC_ERRORS = 3      # 連續失敗幾次就停

def need_upload(name):
    p = f"{ROOT}/{name}"
    if not os.path.exists(p):
        print(f"請上傳 {name}")
        up = files.upload()
        for k, v in up.items():
            open(f"{ROOT}/{name}", "wb").write(v)
    return p

PRESETS = json.load(open(need_upload("presets_prompts.json"), encoding="utf-8"))
SKIP_KEYS = set()
if input("有 agy_manifest.json 要跳過 AGY 已有圖的 preset 嗎？(y/N) ").strip().lower() == "y":
    am = json.load(open(need_upload("agy_manifest.json"), encoding="utf-8"))
    SKIP_KEYS = {k for k, v in am.items() if v.get("status") == "ok"}
print(len(PRESETS), "presets；跳過", len(SKIP_KEYS), "個 AGY 已有圖者")

In [ ]:
# 3) 輔助函式（純 Python，不依賴 GPU）
def parse_ratio(prompt):
    m = re.search(r"aspect ratio\s*(\d+)\s*:\s*(\d+)", prompt)
    return (int(m.group(1)), int(m.group(2))) if m else (1, 1)

def size_for(ratio, max_pixels=MAX_PIXELS, long_cap=1536, mult=16):
    rw, rh = ratio
    h = (max_pixels * rh / rw) ** 0.5
    w = h * rw / rh
    scale = min(1.0, long_cap / max(w, h))
    w, h = w * scale, h * scale
    return max(mult, int(round(w / mult)) * mult), max(mult, int(round(h / mult)) * mult)

def safe(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def filename_for(p):
    return f"i{int(p['index']):03d}__{safe(p['key'])}.png"

def manifest_key(p):
    return f"{p['group']}/{p['key']}"

def load_manifest():
    if os.path.exists(MANIFEST):
        try:
            return json.load(open(MANIFEST, encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_manifest(m):
    tmp = MANIFEST + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=2)
    os.replace(tmp, MANIFEST)

def sha(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()

In [ ]:
# 4) 載入 SDXL base 1.0（fp16，T4 放得下；免登入）
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_ID = "madebyollin/sdxl-vae-fp16-fix"   # 避免 fp16 VAE 產生全黑圖；同樣免登入
vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.to("cuda")
pipe.enable_vae_tiling()
print("模型已載入")

In [ ]:
# 5) 單張生成函式與煙霧測試
STEPS, CFG = 25, 7.0
NEGATIVE = "blurry, low quality, watermark, signature, logo, extra text, deformed"

def clean_prompt(prompt):
    """只用於送進模型：去掉 markdown 記號與章節標頭，讓 77 tokens 裝更多內容。原 prompt 與 sha 不變。"""
    t = re.sub(r"aspect ratio\s*\d+\s*:\s*\d+.*", "", prompt)
    t = re.sub(r"---.*?---", " ", t)
    t = re.sub(r"\[Section \d+\]", " ", t)
    t = re.sub(r"[*#|\[\]]+", " ", t)
    t = re.sub(r"\b(Visual|Param|Subject|Title|Body|Art Styles & Tech|Text Language|Quality)\s*:", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def generate_one(p, seed=None):
    prompt = clean_prompt(p["prompt"])
    w, h = size_for(parse_ratio(p["prompt"]))
    n_tok = len(pipe.tokenizer(prompt, add_special_tokens=True).input_ids)
    gen = torch.Generator("cpu").manual_seed(int(p["index"]) if seed is None else seed)
    t0 = time.time()
    img = pipe(prompt=prompt, negative_prompt=NEGATIVE, width=w, height=h, num_inference_steps=STEPS,
               guidance_scale=CFG, generator=gen).images[0]
    return img, {"width": w, "height": h, "seconds": round(time.time() - t0, 1), "tokens": n_tok, "truncated": n_tok > 77}

p0 = PRESETS[0]
img, info = generate_one(p0)
print(p0["name"], info, f"顯存峰值 {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")
img.resize((img.width // 2, img.height // 2))   # 預覽（只縮顯示，不縮存檔）

In [ ]:
# 6) 批次（可續跑）：已在 manifest 為 ok 且檔案存在者跳過；撞錯記錄後繼續
from PIL import Image
manifest = load_manifest()
new_done = errors_in_row = 0
t_start = time.time()
for p in PRESETS:
    k = manifest_key(p)
    fn = filename_for(p)
    path = f"{ORIG}/{fn}"
    if k in SKIP_KEYS:
        continue
    if manifest.get(k, {}).get("status") == "ok" and os.path.exists(path):
        continue
    if new_done >= MAX_NEW_PER_RUN:
        print("已達本次上限，停止。再執行本格會續做。"); break
    try:
        img, info = generate_one(p)
        img.save(path, format="PNG")
        with Image.open(path) as chk:
            assert chk.size == (info["width"], info["height"])
        manifest[k] = {"original": f"preset-originals/colab/{fn}", "width": info["width"], "height": info["height"],
                       "bytes": os.path.getsize(path), "prompt_sha256": sha(p["prompt"]),
                       "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                       "status": "ok", "watermark": False, "model": "SDXL-base-1.0",
                       "truncated": info["truncated"]}
        new_done += 1; errors_in_row = 0
        print(f"[{p['index']:>3}] ok {info['seconds']}s {info['width']}x{info['height']} {k}")
    except Exception as e:
        errors_in_row += 1
        manifest[k] = {"original": None, "prompt_sha256": sha(p["prompt"]), "status": "error", "error": f"{type(e).__name__}: {e}"[:300]}
        print(f"[{p['index']:>3}] ERROR {type(e).__name__}: {str(e)[:120]}")
        torch.cuda.empty_cache()
        if errors_in_row >= MAX_CONSEC_ERRORS:
            print("連續失敗，停止；請先查原因再重跑。"); save_manifest(manifest); break
    save_manifest(manifest)
ok = sum(1 for v in manifest.values() if v.get("status") == "ok")
print(f"本次新增 {new_done} 張；累計 ok {ok}／{len(PRESETS)}；耗時 {(time.time()-t_start)/60:.1f} 分")

In [ ]:
# 7) 打包下載（含 originals 與 manifest）
import shutil
shutil.make_archive("/content/colab_preset_images", "zip", ROOT, ".")
files.download("/content/colab_preset_images.zip")